# Real TCGA-LUSC data for reviewer comments 2 and 3

This notebook needs no account, no token and no manifest. Everything comes from the open-access GDC API.

1. **Comment 2.** It downloads the open-access *Masked Somatic Mutation* files for TCGA-LUSC and computes TMB and EGFR, STK11, KRAS, KEAP1 and TP53 status for each case. It also pulls smoking exposure.
2. **Comment 3.** It downloads diagnostic whole-slide images one at a time and runs a pretrained pathology foundation model on tissue patches. Each slide is deleted once it is processed, so disk use stays small.

**How to run:** Runtime > Change runtime type > T4 GPU > Save. Then Runtime > Run all. At the end a file called `tcga_lusc_real_outputs.zip` downloads automatically. Send that file back.

Expected time with a T4 GPU and 30 slides: about 30 to 60 minutes.

In [ ]:
# Settings. The defaults are fine; change only if needed.
N_SLIDES      = 30        # number of diagnostic slides to embed
MAX_SLIDE_GB  = 1.5       # skip very large slides to keep download time down
MAX_PATCHES   = 1000      # tissue patches per slide (random sample, fixed seed)
MODEL         = "phikon"  # "phikon" (open, no login) or "uni" (needs Hugging Face approval and login)
SEED          = 2026
EXOME_MB      = 38.0      # TMB denominator in megabases (exome convention); raw counts are also saved

In [ ]:
!pip -q install openslide-python openslide-bin timm transformers pandas requests

In [ ]:
import io, os, json, time, tarfile, gzip, glob, shutil, random
import numpy as np, pandas as pd, requests, torch

API = "https://api.gdc.cancer.gov"
OUT = "tcga_lusc_real_outputs"
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

def gdc_files(filters, fields, size=5000):
    r = requests.post(f"{API}/files", json={"filters": filters, "fields": ",".join(fields),
                                           "format": "JSON", "size": size}, timeout=120)
    r.raise_for_status()
    return r.json()["data"]["hits"]

def f_and(*ops):
    return {"op": "and", "content": list(ops)}

def f_in(field, values):
    return {"op": "in", "content": {"field": field, "value": values}}

LUSC = f_in("cases.project.project_id", ["TCGA-LUSC"])
OPEN = f_in("access", ["open"])

## Part 1 (comment 2): TMB and driver-gene status from masked somatic MAFs

In [ ]:
hits = gdc_files(f_and(LUSC, OPEN, f_in("data_type", ["Masked Somatic Mutation"]), f_in("data_format", ["MAF"])),
                 ["file_id", "file_name", "file_size", "cases.case_id", "cases.submitter_id",
                  "cases.samples.sample_type", "cases.samples.submitter_id"])
maf_index = pd.DataFrame([{
    "file_id": h["file_id"], "file_name": h["file_name"], "file_size": h["file_size"],
    "case_id": h["cases"][0]["case_id"], "case_submitter_id": h["cases"][0]["submitter_id"],
    "sample_types": ";".join(sorted({s.get("sample_type", "") for c in h["cases"] for s in c.get("samples", [])})),
} for h in hits])
print(len(maf_index), "MAF files for", maf_index.case_id.nunique(), "cases")
maf_index.to_csv(f"{OUT}/maf_file_index.csv", index=False)

In [ ]:
os.makedirs("mafs", exist_ok=True)
ids = maf_index.file_id.tolist()
for i in range(0, len(ids), 100):
    r = requests.post(f"{API}/data", json={"ids": ids[i:i+100]}, timeout=600)
    r.raise_for_status()
    try:
        with tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz") as t:
            t.extractall("mafs")
    except tarfile.ReadError:  # a batch of one file comes back as the file itself
        one = maf_index[maf_index.file_id == ids[i]].file_name.iloc[0]
        os.makedirs(f"mafs/{ids[i]}", exist_ok=True)
        open(f"mafs/{ids[i]}/{one}", "wb").write(r.content)
    print("downloaded", min(i+100, len(ids)), "/", len(ids))

In [ ]:
NONSYN = {"Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del", "Frame_Shift_Ins", "In_Frame_Del",
          "In_Frame_Ins", "Splice_Site", "Translation_Start_Site", "Nonstop_Mutation"}
GENES = ["EGFR", "STK11", "KRAS", "KEAP1", "TP53"]
id_by_name = dict(zip(maf_index.file_name, maf_index.file_id))
case_by_id = dict(zip(maf_index.file_id, maf_index.case_id))
sub_by_id = dict(zip(maf_index.file_id, maf_index.case_submitter_id))

rows = []
for f in glob.glob("mafs/**/*.maf.gz", recursive=True):
    parent = os.path.basename(os.path.dirname(f))
    fid = parent if parent in case_by_id else id_by_name.get(os.path.basename(f))
    if fid is None:
        continue
    m = pd.read_csv(f, sep="\t", comment="#", low_memory=False,
                    usecols=["Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode"])
    ns = m[m.Variant_Classification.isin(NONSYN)]
    bc = str(m.Tumor_Sample_Barcode.iloc[0]) if len(m) else ""
    row = {"file_id": fid, "case_id": case_by_id[fid], "case_submitter_id": sub_by_id[fid],
           "tumor_sample_barcode": bc, "n_variants_all": len(m), "n_nonsynonymous": len(ns),
           "tmb_per_mb": len(ns) / EXOME_MB}
    for g in GENES:
        row[f"{g.lower()}_mut"] = int((ns.Hugo_Symbol == g).any())
    rows.append(row)

per_file = pd.DataFrame(rows)
per_file["is_primary"] = per_file.tumor_sample_barcode.str[13:15].eq("01")
per_file = per_file.sort_values(["case_id", "is_primary", "n_variants_all"], ascending=[True, False, False])
per_file.to_csv(f"{OUT}/mutation_per_file.csv", index=False)
mut = per_file.drop_duplicates("case_id").drop(columns=["file_id"])  # one tumour per case, primary first
mut.to_csv(f"{OUT}/tcga_lusc_mutation_features.csv", index=False)
print(len(mut), "cases with mutation data")
print(mut[["tmb_per_mb"] + [f"{g.lower()}_mut" for g in GENES]].describe().T[["count", "mean", "50%"]])

In [ ]:
# Smoking exposure (fills the 'Smoking history' gap in the schema-coverage table)
r = requests.post(f"{API}/cases", json={"filters": f_in("project.project_id", ["TCGA-LUSC"]),
    "fields": "case_id,submitter_id,exposures.tobacco_smoking_status,exposures.pack_years_smoked,exposures.cigarettes_per_day",
    "format": "JSON", "size": 5000}, timeout=120)
r.raise_for_status()
ex = []
for h in r.json()["data"]["hits"]:
    e = (h.get("exposures") or [{}])[0]
    ex.append({"case_id": h["case_id"], "case_submitter_id": h["submitter_id"],
               "tobacco_smoking_status": e.get("tobacco_smoking_status"),
               "pack_years_smoked": e.get("pack_years_smoked"),
               "cigarettes_per_day": e.get("cigarettes_per_day")})
pd.DataFrame(ex).to_csv(f"{OUT}/tcga_lusc_exposure.csv", index=False)
print(pd.DataFrame(ex).tobacco_smoking_status.value_counts(dropna=False))

## Part 2 (comment 3): pathology foundation model on real diagnostic slides

In [ ]:
hits = gdc_files(f_and(LUSC, OPEN, f_in("data_type", ["Slide Image"]), f_in("experimental_strategy", ["Diagnostic Slide"])),
                 ["file_id", "file_name", "file_size", "cases.case_id", "cases.submitter_id"])
slides = pd.DataFrame([{"file_id": h["file_id"], "file_name": h["file_name"], "file_size": h["file_size"],
                        "case_id": h["cases"][0]["case_id"], "case_submitter_id": h["cases"][0]["submitter_id"]} for h in hits])
print(len(slides), "open diagnostic slides for", slides.case_id.nunique(), "cases")
slides.to_csv(f"{OUT}/slide_file_index_all.csv", index=False)
elig = slides[slides.file_size < MAX_SLIDE_GB * 1e9].drop_duplicates("case_id")
chosen = elig.sample(n=min(N_SLIDES, len(elig)), random_state=SEED).reset_index(drop=True)
chosen.to_csv(f"{OUT}/slides_selected.csv", index=False)
print(len(chosen), "slides selected; total", round(chosen.file_size.sum() / 1e9, 1), "GB")

In [ ]:
import timm
from PIL import Image
import openslide

if MODEL == "phikon":
    from transformers import ViTModel
    net = ViTModel.from_pretrained("owkin/phikon", add_pooling_layer=False).to(DEV).eval()
    MEAN, STD, DIM = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225), 768
    LABEL = "Phikon (ViT-B/16, iBOT, Filiot et al. 2023), owkin/phikon"
    embed = lambda x: net(pixel_values=x).last_hidden_state[:, 0, :]
elif MODEL == "uni":
    from huggingface_hub import login; login()
    net = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True).to(DEV).eval()
    MEAN, STD, DIM = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225), 1024
    LABEL = "UNI (ViT-L/16, Chen et al. 2024), MahmoodLab/uni"
    embed = lambda x: net(x)
mt = torch.tensor(MEAN, device=DEV).view(1, 3, 1, 1)
st = torch.tensor(STD, device=DEV).view(1, 3, 1, 1)
print("model:", LABEL)

In [ ]:
PATCH_PX, TARGET_MPP, BATCH = 224, 0.5, 64

def download(file_id, path, size, tries=8):
    # Resumes from where a dropped connection stopped (HTTP Range), with retries.
    for attempt in range(1, tries + 1):
        have = os.path.getsize(path) if os.path.exists(path) else 0
        if size and have >= size:
            return
        hdr = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(f"{API}/data/{file_id}", headers=hdr, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                mode = "ab" if (have and r.status_code == 206) else "wb"
                with open(path, mode) as fh:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        fh.write(chunk)
            if not size or os.path.getsize(path) >= size:
                return
        except Exception as e:
            print(f"   download interrupted ({type(e).__name__}), retry {attempt}/{tries}", flush=True)
            time.sleep(min(60, 10 * attempt))
    raise RuntimeError("download failed after retries")

def tissue_coords(sl, max_patches, sat_thresh=20):
    mpp = float(sl.properties.get("openslide.mpp-x", 0.25))
    p0 = int(round(PATCH_PX * TARGET_MPP / mpp))
    W, H = sl.dimensions
    thumb = sl.get_thumbnail((W // 64, H // 64))
    sat = np.array(thumb.convert("HSV"))[:, :, 1]
    mask = sat > sat_thresh
    sx, sy = W / mask.shape[1], H / mask.shape[0]
    coords = []
    for y in range(0, H - p0, p0):
        for x in range(0, W - p0, p0):
            y0, x0 = int(y / sy), int(x / sx)
            y1, x1 = max(y0 + 1, int((y + p0) / sy)), max(x0 + 1, int((x + p0) / sx))
            if mask[y0:y1, x0:x1].mean() > 0.5:
                coords.append((x, y))
    n_tissue = len(coords)
    if n_tissue > max_patches:
        rng = np.random.default_rng(0)
        coords = [coords[i] for i in sorted(rng.choice(n_tissue, max_patches, replace=False))]
    return coords, p0, mpp, n_tissue, thumb

for d in ["patch_embeddings", "patch_coords", "thumbnails"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

# Keep slides already finished in an earlier run
if os.path.exists(f"{OUT}/run_log.json") and os.path.exists(f"{OUT}/slide_embeddings.csv"):
    log = json.load(open(f"{OUT}/run_log.json"))
    srows = pd.read_csv(f"{OUT}/slide_embeddings.csv").to_dict("records")
else:
    log = {"model": LABEL, "device": DEV, "patch_px": PATCH_PX, "target_mpp": TARGET_MPP,
           "max_patches": MAX_PATCHES, "seed": SEED, "slides": []}
    srows = []
log.setdefault("failed", [])
done = {r["slide_id"] for r in srows}

for k, s in chosen.iterrows():
    sid = s.file_name.replace(".svs", "")
    if sid in done:
        print(f"[{k+1}/{len(chosen)}] {sid}: already done", flush=True)
        continue
    t0 = time.time()
    path = f"/content/{s.file_name}"
    try:
        download(s.file_id, path, int(s.file_size))
        t_dl = time.time() - t0
        sl = openslide.OpenSlide(path)
        coords, p0, mpp, n_tissue, thumb = tissue_coords(sl, MAX_PATCHES)
        feats = []
        for i in range(0, len(coords), BATCH):
            imgs = [np.asarray(sl.read_region(c, 0, (p0, p0)).convert("RGB").resize((PATCH_PX, PATCH_PX), Image.BILINEAR))
                    for c in coords[i:i + BATCH]]
            x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float().to(DEV) / 255.0
            with torch.inference_mode(), torch.autocast(device_type=DEV, enabled=(DEV == "cuda")):
                feats.append(embed((x - mt) / st).float().cpu().numpy())
        E = np.concatenate(feats) if feats else np.zeros((0, DIM), np.float32)
        sl.close()
    except Exception as e:
        print(f"[{k+1}/{len(chosen)}] {sid}: SKIPPED ({type(e).__name__}: {e})", flush=True)
        log["failed"].append({"slide_id": sid, "error": f"{type(e).__name__}: {e}"})
        json.dump(log, open(f"{OUT}/run_log.json", "w"), indent=2)
        if os.path.exists(path):
            os.remove(path)
        continue
    np.save(f"{OUT}/patch_embeddings/{sid}.npy", E.astype(np.float16))
    np.save(f"{OUT}/patch_coords/{sid}.npy", np.array(coords, dtype=np.int32))
    if len(srows) < 6:
        thumb.convert("RGB").save(f"{OUT}/thumbnails/{sid}.jpg", quality=85)
    srows.append({"slide_id": sid, "file_id": s.file_id, "case_id": s.case_id,
                  "case_submitter_id": s.case_submitter_id, "n_patches": len(coords),
                  **{f"e{j}": v for j, v in enumerate(E.mean(0) if len(E) else np.zeros(DIM))}})
    log["slides"].append({"slide_id": sid, "case_id": s.case_id, "file_gb": round(s.file_size / 1e9, 2),
                          "slide_mpp": mpp, "level0_patch_px": p0, "tissue_patches_available": n_tissue,
                          "patches_embedded": len(coords), "download_s": round(t_dl, 1),
                          "total_s": round(time.time() - t0, 1)})
    os.remove(path)
    pd.DataFrame(srows).to_csv(f"{OUT}/slide_embeddings.csv", index=False)
    json.dump(log, open(f"{OUT}/run_log.json", "w"), indent=2)
    print(f"[{k+1}/{len(chosen)}] {sid}: {len(coords)} patches, {time.time()-t0:.0f}s", flush=True)
print("done:", len(srows), "slides embedded,", len(log["failed"]), "skipped")

## Package and download the outputs

In [ ]:
shutil.make_archive("tcga_lusc_real_outputs", "zip", OUT)
print(round(os.path.getsize("tcga_lusc_real_outputs.zip") / 1e6, 1), "MB")
from google.colab import files
files.download("tcga_lusc_real_outputs.zip")